In [ ]:
##ttfs encoding method
####Tictactoe reference code : https://codelearn.io/sharing/day-ai-danh-tictactoe-voi-deep-learning
##DSQN reference code : https://github.com/mahmoudakl/dsrl
##wandb link: https://wandb.ai/kradeero-ohio-university/experiments/runs/6ae7xe4w 


import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import wandb
import os

class BaseModel:
    """Base class for reinforcement learning models"""
    def __init__(self, discount_factor, epsilon, e_min, e_max):
        """
        Initialize base parameters for reinforcement learning models
        
        Args:
            discount_factor (float): Discount factor for future rewards (gamma)
            epsilon (float): Initial exploration rate
            e_min (int): Minimum experiences before training starts
            e_max (int): Maximum experience replay buffer size
        """
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.e_max = e_max

class Tictactoe_v0:
    def __init__(self):
        self.board = [0] * 9
        self.wining_position = [[0, 1, 2], [3, 4, 5], [6, 7, 8],
                                [0, 3, 6], [1, 4, 7], [2, 5, 8],
                                [0, 4, 8], [6, 4, 2]]
        self.current_turn = 1
        self.player_mark = 1

    def reset(self, is_human_first):
        self.board = [0] * 9
        self.current_turn = 1
        self.player_mark = 1 if is_human_first else -1
        if not is_human_first:
            self.env_act()
        return self.board.copy()

    def check_win(self):
        for pst in self.wining_position:
            if str(self.board[pst[0]]) + str(self.board[pst[1]]) + str(self.board[pst[2]]) in ['111', '-1-1-1']:
                if self.current_turn == self.player_mark:
                    return 1, True #Player win
                return -1, True #AI win
        if 0 not in self.board:
            return 0, True #Draw
        return 0, False #Continue

    def env_act(self):
        action = random.choice([i for i in range(len(self.board)) if self.board[i] == 0])
        for pst in self.wining_position:
            com = str(self.board[pst[0]]) + str(self.board[pst[1]]) + str(self.board[pst[2]])
            if com.replace('0', '') == str(self.current_turn) * 2:
                if self.board[pst[0]] == 0:
                    action = pst[0]
                elif self.board[pst[1]] == 0:
                    action = pst[1]
                else:
                    action = pst[2]
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        reward, done = self.check_win()
        self.current_turn = self.current_turn * -1
        return reward, done

    def step(self, action):
        if self.board[action] != 0:
            raise Exception('Invalid action')
        self.board[action] = self.current_turn
        reward, done = self.check_win()
        self.current_turn = self.current_turn * -1
        if not done:
            reward, done = self.env_act()
        return self.board.copy(), reward, done, None
    
    def render(self):
        """Visualize the board with X/O positions"""
        symbols = {1: 'X', -1: 'O', 0: ' '}
        print("\nCurrent Board:")
        for i in range(3):
            print(f" {symbols[self.board[i*3]]} | {symbols[self.board[i*3+1]]} | {symbols[self.board[i*3+2]]} ")
            if i < 2: print("-----------")
        print()

class EpsilonGreedy:
    def __init__(self, epsilon):
        self.epsilon = epsilon

    def perform(self, q_value, action_space: list = None):
        prob = np.random.sample()
        if prob <= self.epsilon:
            if action_space is None:
                return np.random.randint(len(q_value))
            return np.random.choice(action_space)
        else:
            if action_space is None:
                return np.argmax(q_value)
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1]

    def decay(self, decay_value, lower_bound):
        self.epsilon = max(self.epsilon * decay_value, lower_bound)

class ExperienceReplay:
    def __init__(self, e_max: int):
        if e_max <= 0:
            raise ValueError('Invalid value for memory size')
        self.e_max = e_max
        self.memory = list()
        self.index = 0

    def add_experience(self, sample: list):
        if len(sample) != 5:
            raise Exception('Invalid sample')
        if len(self.memory) < self.e_max:
            self.memory.append(sample)
        else:
            self.memory[self.index] = sample
        self.index = (self.index + 1) % self.e_max

    def sample_experience(self, sample_size: int, cer_mode: bool):
        samples = random.sample(self.memory, sample_size)
        if cer_mode:
            samples[-1] = self.memory[self.index - 1]
        s_batch, a_batch, r_batch, ns_batch, done_batch = map(np.array, zip(*samples))
        return s_batch, a_batch, r_batch, ns_batch, done_batch

    def get_size(self):
        return len(self.memory)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn, mem, spk = [], [], []
        
        for l in range(len(self.weights)):
            syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            spk.append([])
        
        mem_rec = []
        
        for t in range(self.simulation_time):
            input = x[:, t, :]
            
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.mm(input, self.weights[l])
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
        
        q_values = mem[-1]
        return q_values, mem_rec, spk

class DQN(BaseModel):
    """Deep Q-Network agent with Deep Spiking Neural Network"""
    def __init__(self, discount_factor: float, epsilon: float, e_min: int, e_max: int, dsnn_config: dict):
        super().__init__(discount_factor, epsilon, e_min, e_max)
        self.gamma = discount_factor
        self.epsilon_greedy = EpsilonGreedy(epsilon)
        self.e_min = e_min
        self.exp_replay = ExperienceReplay(e_max)
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        self.cache = []
        self.initial_lr = dsnn_config['learning_rate']  # Store initial learning rate

    def ttfs_encode(self, state):
        """Convert board state to Time-To-First-Spike encoding"""
        encoded = torch.zeros(self.simulation_time, 9, device=device)
        for pos, val in enumerate(state):
            # Map board values to spike times (earlier spike = stronger signal)
            if val == 1:  # 'X': spike at t=0 (earliest)
                spike_time = 0
            elif val == -1:  # 'O': spike at t=1
                spike_time = 3
            else:  # Empty: spike at t=9 (late, near end of simulation)
                spike_time = 9
            
            # Add small random noise to spike time to avoid ties
            noise = np.random.normal(0, 0.05)
            spike_time = max(0, min(self.simulation_time - 1, int(spike_time + noise)))
            
            # Set spike at the determined timestep
            encoded[spike_time, pos] = 1.0
        
        return encoded.unsqueeze(0)

    def observe(self, state, action_space: list = None):
        """Get best action for given state (exploitation)"""
        with torch.no_grad():
            encoded_state = self.ttfs_encode(state)
            q_values, _, _ = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
        if action_space is not None:
            valid_q = [(q_values[a], a) for a in action_space]
            return max(valid_q, key=lambda x: x[0])[1]
        return np.argmax(q_values)

    def observe_on_training(self, state, action_space: list = None) -> int:
        """Get action with epsilon-greedy exploration"""
        with torch.no_grad():
            encoded_state = self.ttfs_encode(state)
            q_values, _, _ = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
        action = self.epsilon_greedy.perform(q_values, action_space)
        self.cache.extend([state, action])
        return action

    def take_reward(self, reward, next_state, done):
        """Store experience in replay buffer"""
        self.cache.extend([reward, next_state, done])
        self.exp_replay.add_experience(self.cache.copy())
        self.cache.clear()

    def train_network(self, sample_size: int, batch_size: int):
        """Train network with TTFS-encoded experiences"""
        if self.exp_replay.get_size() < self.e_min:
            return None
        states, actions, rewards, next_states, dones = self.exp_replay.sample_experience(sample_size, cer_mode=False)
        state_batch = torch.stack([self.ttfs_encode(s) for s in states]).squeeze(1)
        next_state_batch = torch.stack([self.ttfs_encode(ns) for ns in next_states]).squeeze(1)
        action_batch = torch.LongTensor(actions).to(device)
        reward_batch = torch.FloatTensor(rewards).to(device)
        done_batch = torch.BoolTensor(dones).to(device)
        with torch.no_grad():
            next_q_values, _, _ = self.target_net(next_state_batch)
            # Normalize next Q-values
            max_abs_next_q = torch.max(torch.abs(next_q_values))
            if max_abs_next_q > 1e-6:  # Avoid division by zero
                next_q_values = next_q_values / max_abs_next_q
            max_next_q = next_q_values.max(1)[0]
            # Clip target Q-values to reward range [-1, 1]
            target_q = reward_batch + (1 - done_batch.float()) * self.gamma * max_next_q
            target_q = torch.clamp(target_q, min=-1.0, max=1.0)
        current_q, mem_rec, spk_rec = self.training_net(state_batch)
        # Normalize current Q-values
        max_abs_current_q = torch.max(torch.abs(current_q))
        if max_abs_current_q > 1e-6:  # Avoid division by zero
            current_q = current_q / max_abs_current_q
        current_q = current_q.gather(1, action_batch.unsqueeze(1)).squeeze(1)
        self.training_net.optimizer.zero_grad()
        loss = F.mse_loss(current_q, target_q)
        # Clip loss to prevent large gradients
        loss = torch.clamp(loss, min=-100.0, max=100.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()
        return loss.item()

    def update_target_network(self):
        """Update target network with training network weights"""
        self.target_net.load_state_dict(self.training_net.state_dict())

    def save_model(self, filename):
        """Save model weights to file"""
        torch.save({
            'training_net': self.training_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'epsilon': self.epsilon_greedy.epsilon
        }, filename)

    def load_model(self, filename):
        """Load model weights from file"""
        checkpoint = torch.load(filename)
        self.training_net.load_state_dict(checkpoint['training_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])
        self.epsilon_greedy.epsilon = checkpoint['epsilon']

def print_spikes(spk_rec, timestep=-1):
    """Print spike activity for last timestep"""
    print("\nSpike Activity:")
    for layer_idx, layer_spikes in enumerate(spk_rec):
        if len(layer_spikes) > 0:
            spike_count = layer_spikes[timestep].sum().item()
            print(f"Layer {layer_idx+1}: {spike_count} spikes")

def print_all_spikes(spk_rec):
    for layer_idx, layer_spikes_list in enumerate(spk_rec):
        if not layer_spikes_list:
            print(f"[Debug] Layer {layer_idx+1} has no spike data.")
            continue
        print(f"\n[Debug] Layer {layer_idx+1} Spikes:")
        layer_spikes_tensor = torch.stack(layer_spikes_list, dim=0)
        print(layer_spikes_tensor)

def print_encoded_state(encoded_state, timesteps=1):
    """Print TTFS encoding for all timesteps"""
    print("\nTTFS Encoding (All Timesteps):")
    encoded_np = encoded_state.squeeze(0).cpu().numpy()
    for t in range(encoded_np.shape[0]):
        print(f"Timestep {t+1}: {encoded_np[t]}")

def print_membrane_potentials(q_values, action):
    """Show output layer decision process"""
    print("\nOutput Membrane Potentials (Q-values):")
    q_np = q_values.detach().cpu().numpy().flatten()
    for i in range(9):
        print(f"Position {i}: {q_np[i]:.2f}")
    print(f"Selected Action: Position {action} (Q-value: {q_np[action]:.2f})")

dsnn_config = {
    'architecture': [9, 128, 128, 9],
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.85,
    'weight_scale': 0.15,
    'batch_size': 32,
    'threshold': 0.1,
    'simulation_time': 5,
    'learning_rate': 0.0001,
    'reset_potential': 0.0
}

env = Tictactoe_v0()
agent = DQN(
    discount_factor=0.95,
    epsilon=1.0,
    e_min=1000,
    e_max=100000,
    dsnn_config=dsnn_config
)

agent.update_target_network()

num_episodes = 20001
batch_size = 32
epochs = 1
sample_size = 64
# lr_decay_factor = 0.9  # Learning rate decay factor
# lr_decay_interval = 1000  # Decay every 1000 episodes
# consecutive_high_loss_count = 0  # For early stopping
# max_consecutive_high_loss = 3  # Stop if loss > 50 for 3 intervals
# loss_threshold = 50.0  # Threshold for high loss

total_loss = 0
episode_count_for_avg_loss = 0
win_count = 0
loss_count = 0
draw_count = 0
total_spike_count = 0
episode_count_for_avg_spikes = 0

wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="ttfs_ttt")

# Training loop
for episode in range(num_episodes):
    state = env.reset(is_human_first=True)
    done = False
    episode_reward = 0

    # # Adjust learning rate if at decay interval
    # if episode > 0 and episode % lr_decay_interval == 0:
    #     for param_group in agent.training_net.optimizer.param_groups:
    #         param_group['lr'] *= lr_decay_factor
    #     wandb.log({"Learning Rate": param_group['lr']})

    # #uncomment the below
    # print(f"\n--- Episode {episode + 1} ---")
    # env.render()
    
    while not done:
        action_space = [i for i, val in enumerate(state) if val == 0]
        action = agent.observe_on_training(state, action_space)
        next_state, reward, done, _ = env.step(action)
        episode_reward += reward

        
        if episode % 100 == 0:
            encoded_state = agent.ttfs_encode(state)
            # #uncomment the below
            # print_encoded_state(encoded_state)
            # env.render()
            with torch.no_grad():
                q_values, mem_rec, spk_rec = agent.training_net(encoded_state)
            step_spike_count = 0
            for layer_spikes in spk_rec:
                if layer_spikes:
                    for t in range(len(layer_spikes)):
                        step_spike_count += layer_spikes[t].sum().item()
            total_spike_count += step_spike_count
            episode_count_for_avg_spikes += 1

            # Show membrane potentials
            # #uncomment the below
            # print_membrane_potentials(q_values[0], action)
            # env.render()
           
            # Show spike activity
            # #uncomment the below
            # print_spikes(spk_rec)
            # print_all_spikes(spk_rec)
            # # wandb.log({"Spikes after Each episode":spk_rec})
            
        agent.take_reward(reward, next_state, done)

        if agent.exp_replay.get_size() > agent.e_min:
            loss = agent.train_network(sample_size, batch_size)
            if loss is not None:
                total_loss += loss
                episode_count_for_avg_loss += 1
            agent.update_target_network()

        state = next_state
    
    agent.epsilon_greedy.decay(decay_value=0.995, lower_bound=0.01)

    if episode_reward == 1:
        win_count += 1
    elif episode_reward == -1:
        loss_count += 1
    else:
        if done:
            draw_count += 1

    combined_rate = win_count + draw_count

    if episode % 100 == 0:
        avg_loss = total_loss / episode_count_for_avg_loss if episode_count_for_avg_loss > 0 else 0
        avg_spike_count = total_spike_count / episode_count_for_avg_spikes if episode_count_for_avg_spikes > 0 else 0
        print(f"Episode Summary {episode}, Games Won: {win_count}, Games Lost: {loss_count}, Games Drawn: {draw_count}")
        print(f"Episode: {episode}, Avg Loss: {avg_loss:.4f}, Avg Spikes: {avg_spike_count:.2f}")

        # # Early stopping check
        # if avg_loss > loss_threshold:
        #     consecutive_high_loss_count += 1
        #     if consecutive_high_loss_count >= max_consecutive_high_loss:
        #         print(f"Early stopping triggered at episode {episode} due to high loss.")
        #         break
        # else:
        #     consecutive_high_loss_count = 0

        wandb.log({
            "Episode": episode,
            "Win Rate": win_count,
            "Loss Rate": loss_count,
            "Draw Rate": draw_count,
            "Win + Draw Rate": combined_rate,
            "Average Loss": avg_loss,
            "Average Spikes": avg_spike_count
        })

        total_loss = 0.0
        episode_count_for_avg_loss = 0
        win_count = 0
        loss_count = 0
        draw_count = 0
        total_spike_count = 0
        episode_count_for_avg_spikes = 0

# Save model with proper path handling
filename = "../saved_models/ttfs_ttt.pth"
agent.save_model(filename)
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/saideepa0501/.netrc
wandb: Currently logged in as: kradeero (kradeero-ohio-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Episode Summary 0, Games Won: 0, Games Lost: 1, Games Drawn: 0
Episode: 0, Avg Loss: 0.0000, Avg Spikes: 153.33
Episode Summary 100, Games Won: 31, Games Lost: 65, Games Drawn: 4
Episode: 100, Avg Loss: 0.0000, Avg Spikes: 199.00
Episode Summary 200, Games Won: 30, Games Lost: 64, Games Drawn: 6
Episode: 200, Avg Loss: 0.0000, Avg Spikes: 155.00
Episode Summary 300, Games Won: 46, Games Lost: 52, Games Drawn: 2
Episode: 300, Avg Loss: 0.3044, Avg Spikes: 157.33
Episode Summary 400, Games Won: 62, Games Lost: 32, Games Drawn: 6
Episode: 400, Avg Loss: 0.2078, Avg Spikes: 179.33
Episode Summary 500, Games Won: 70, Games Lost: 23, Games Drawn: 7
Episode: 500, Avg Loss: 0.1582, Avg Spikes: 146.33
Episode Summary 600, Games Won: 70, Games Lost: 26, Games Drawn: 4
Episode: 600, Avg Loss: 0.1346, Avg Spikes: 151.67
Episode Summary 700, Games Won: 73, Games Lost: 19, Games Drawn: 8
Episode: 700, Avg Loss: 0.1195, Avg Spikes: 165.00
Episode Summary 800, Games Won: 85, Games Lost: 10, Games Draw

Average Loss,█▇▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▁
Average Spikes,▅▃▆█▃▃▃▃▄▄▃▂▂▂▇▄▂▄▃▄▃▃▂▄▅▄▄▁▁▁▃▆▃▁▂▆▄▁▂▂
Draw Rate,▂▃▅█▃▅▃▅▃▂▂▁▆▃▁▅▄▄▂▂▄▄▄▂▅▂▇▅▂▃▂▅▃▅▃▃▆▅▄▃
Episode,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████
Loss Rate,█▃▃▂▃▂▂▂▂▂▂▁▁▁▁▂▂▁▁▁▂▂▁▁▁▁▁▁▁▁▂▂▁▁▁▁▁▁▃▁
Win + Draw Rate,▁▆▅▆▇▇██▇▇██████████████████▇███▇██▇▇▇█▇
Win Rate,▁▃▄▅▆▇█▇████▇████▇██▇██▇▇█▇▇▇▇█▇██▇▇█▇█▇
Average Loss,0.01373
Average Spikes,123.33333
Draw Rate,1
Episode,20000


In [1]:
##Evaluation code for the ttfs encoding
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from copy import deepcopy
import os

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# DQN Agent (from population encoding code)
class DQN:
    def __init__(self, discount_factor=0.95, epsilon=0.1, e_min=4096, e_max=1048576):
        self.training_network = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 9)
        ).to(device)
        self.epsilon = epsilon

    def observe(self, state, action_space=None):
        state_tensor = torch.Tensor(np.array([state])).float().to(device)
        q_value = self.training_network(state_tensor).detach().cpu().numpy().ravel()
        # Count MACs for DQN: input_size * output_size per layer
        mac_count = 9 * 128 + 128 * 128 + 128 * 9  # 18688 MACs
        if action_space is not None:
            return max([[q_value[a], a] for a in action_space], key=lambda x: x[0])[1], mac_count
        return np.argmax(q_value), mac_count

    def load_model(self, path):
        try:
            self.training_network.load_state_dict(torch.load(path, map_location=device))
            self.training_network.eval()
            print(f"DQN model loaded from {path}")
        except FileNotFoundError:
            raise FileNotFoundError(f"DQN model file {path} not found")

# SurrGradSpike (same as provided)
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# DSNN (corrected from TTFS training code, with analysis metrics)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn, mem, spk = [], [], []
        
        for l in range(len(self.weights)):
            syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            spk.append([])
        
        mem_rec = []
        all_spikes = []
        ac_count = 0  # Count synaptic fan-out additions
        internal_mac_count = 0  # Count internal state update MACs
        
        for t in range(self.simulation_time):
            input = x[:, t, :]
            layer_spikes = []
            for l in range(len(self.weights)):
                # Synaptic operations (fan-out, counted as additions)
                if l == 0:
                    h = torch.mm(input, self.weights[l])
                    num_spikes = torch.sum(input > 0).item()  # Binarize inputs
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                    num_spikes = torch.sum(spk[l-1][-1]).item()  # Count binary spikes
                ac_count += num_spikes * self.weights[l].size(1)  # ACs: additions for synaptic fan-out
                
                # Internal state updates (count MACs)
                num_neurons = self.weights[l].size(1)
                internal_mac_count += num_neurons * 2  # 1 mult (alpha * syn) + 1 add, 1 mult (beta * mem) + 1 add
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    # Additional MACs for reset (approximated as 2 per neuron if spiking)
                    internal_mac_count += num_neurons * 2 * torch.mean(spk_current).item()  # Approx. 2 MACs per spiking neuron
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                    layer_spikes.append(spk_current)
                else:
                    layer_spikes.append(torch.zeros_like(mem[l]))
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
                
            all_spikes.append(layer_spikes)
        
        q_values = mem[-1]
        return q_values, mem_rec, all_spikes, ac_count, internal_mac_count

# DSQN with TTFS encoding
class DSQN:
    def __init__(self, discount_factor=0.95, epsilon=0.1, e_min=1000, e_max=100000, dsnn_config=None):
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.e_min = e_min
        self.exp_replay = None
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']

        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.population_size = 1  # Not used in encoding

    def ttfs_encode(self, state):
        """Convert board state to Time-To-First-Spike encoding."""
        encoded = torch.zeros(self.simulation_time, 9, device=device)
        for pos, val in enumerate(state):
            # Map board values to spike times (earlier spike = stronger signal)
            if val == 1:  # 'X': spike at t=0 (earliest)
                spike_time = 0
            elif val == -1:  # 'O': spike at t=1
                spike_time = 3
            else:  # Empty: spike at t=4 (late, near end of simulation)
                spike_time = 9
            
            # Add small random noise to spike time to avoid ties
            noise = np.random.normal(0, 0.1)
            spike_time = max(0, min(self.simulation_time - 1, int(spike_time + noise)))
            
            # Set spike at the determined timestep
            encoded[spike_time, pos] = 1.0
        
        return encoded.unsqueeze(0)

    def observe(self, state, action_space=None):
        with torch.no_grad():
            encoded_state = self.ttfs_encode(state)
            q_values, mem_rec, all_spikes, ac_count, internal_mac_count = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
            if action_space is not None:
                valid_q = [(q_values[a], a) for a in action_space]
                action = max(valid_q, key=lambda x: x[0])[1]
            else:
                action = np.argmax(q_values)
            return action, mem_rec, all_spikes, ac_count, internal_mac_count

    def load_model(self, filename):
        try:
            checkpoint = torch.load(filename, map_location=device)
            self.training_net.load_state_dict(checkpoint['training_net'])
            self.target_net.load_state_dict(checkpoint['target_net'])
            self.epsilon = checkpoint['epsilon']
            self.training_net.eval()
            self.target_net.eval()
            print(f"DSQN model loaded from {filename}")
        except FileNotFoundError:
            raise FileNotFoundError(f"DSQN model file {filename} not found")

# Analysis functions (same as previous codes)
def analyze_decision_stabilization(mem_rec, simulation_time):
    decision_times = []
    final_decision = torch.argmax(mem_rec[-1], dim=1)
    for b in range(mem_rec[0].size(0)):
        stabilized = False
        for t in range(simulation_time):
            current_decision = torch.argmax(mem_rec[t], dim=1)[b]
            if current_decision == final_decision[b]:
                stable = True
                for t_next in range(t, simulation_time):
                    if torch.argmax(mem_rec[t_next], dim=1)[b] != final_decision[b]:
                        stable = False
                        break
                if stable:
                    decision_times.append(t + 1)
                    stabilized = True
                    break
        if not stabilized:
            decision_times.append(simulation_time)
    return decision_times

def analyze_spikes(all_spikes, architecture, ac_count, internal_mac_count):
    total_spikes = 0
    total_neurons = sum(architecture[1:-1])
    spike_counts = []
    
    for t in range(len(all_spikes)):
        for l in range(len(all_spikes[t])):
            if l < len(architecture) - 1:
                spikes = all_spikes[t][l]
                spike_count = torch.sum(spikes).item()
                total_spikes += spike_count
                spike_counts.append(spike_count)
    
    sparsity = 1 - (total_spikes / (total_neurons * len(all_spikes)))
    return total_spikes, sparsity, spike_counts, ac_count, internal_mac_count

# Utility functions (same as previous codes)
def print_board(board, move=None, agent=None):
    print("\nBoard state:")
    for row in board:
        print(" | ".join(row))
        print("-" * 9)
    if move and agent:
        print(f"{agent} played at position ({move[0]}, {move[1]})")

def game_over(board):
    winning_positions = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],
        [0, 3, 6], [1, 4, 7], [2, 5, 8],
        [0, 4, 8], [2, 4, 6]
    ]
    for pos in winning_positions:
        if board[pos[0]] == board[pos[1]] == board[pos[2]] != 0:
            return True, 1 if board[pos[0]] == 1 else -1
    if 0 not in board:
        return True, 0
    return False, None

def random_move(board):
    action_space = [i for i, val in enumerate(board) if val == 0]
    return random.choice(action_space)

# Play game function (corrected)
def play_game(agent, agent_first=True, visualize=False, collect_analysis=False):
    flat_board = [0] * 9
    grid_board = [['_' for _ in range(3)] for _ in range(3)]
    current_player = 'x'
    agent_symbol = 'X'
    random_symbol = 'O'

    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    } if collect_analysis else None

    if visualize:
        print("\n=== New Game ===")
        print(f"Agent plays as {agent_symbol}, Random plays as {random_symbol}")
        print_board(grid_board)

    while True:
        if current_player == 'x':
            # Agent plays X
            action_space = [i for i, val in enumerate(flat_board) if val == 0]
            if collect_analysis and isinstance(agent, DSQN):
                action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(flat_board, action_space)
                total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(all_spikes, agent.training_net.architecture, ac_count, internal_mac_count)
                analysis_data['total_spikes'].append(total_spikes)
                analysis_data['sparsity'].append(sparsity)
                analysis_data['ac_counts'].append(ac_count)
                analysis_data['internal_mac_counts'].append(internal_mac_count)
                decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
                analysis_data['decision_times'].extend(decision_times)
            elif collect_analysis and isinstance(agent, DQN):
                action, mac_count = agent.observe(flat_board, action_space)
                analysis_data['mac_counts'].append(mac_count)
            else:
                action = agent.observe(flat_board, action_space)[0]
            flat_board[action] = 1
            row, col = action // 3, action % 3
            grid_board[row][col] = 'x'
            if visualize:
                print_board(grid_board, (row, col), f"Agent ({agent_symbol})")
        else:
            # Random plays O
            action = random_move(flat_board)
            flat_board[action] = -1
            row, col = action // 3, action % 3
            grid_board[row][col] = 'o'
            if visualize:
                print_board(grid_board, (row, col), f"Random ({random_symbol})")

        done, result = game_over(flat_board)
        if done:
            if visualize:
                if result == 0:
                    print("Game ended in a draw!")
                elif result == 1:
                    print(f"Agent ({agent_symbol}) wins!")
                else:
                    print(f"Random ({random_symbol}) wins!")
            if result == 0:
                return 'draw', analysis_data
            elif result == 1:
                return 'agent', analysis_data
            else:
                return 'random', analysis_data

        current_player = 'o' if current_player == 'x' else 'x'

# Test agent vs random
def test_agent_vs_random(agent, agent_name, num_games=100, visualize_all=False):
    results = {'agent': 0, 'random': 0, 'draw': 0}
    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'mac_counts': [],
        'internal_mac_counts': []
    }

    # Agent goes first (as X)
    if isinstance(agent, DQN):
        agent.load_model("../saved_models/ttt_dqn_first_final.pth")
    elif isinstance(agent, DSQN):
        agent.load_model("../saved_models/ttfs_ttt.pth")
    for i in range(num_games):
        visualize = visualize_all
        winner, game_analysis = play_game(agent, agent_first=True, visualize=visualize, collect_analysis=True)
        results[winner] += 1
        if game_analysis:
            analysis_data['total_spikes'].extend(game_analysis['total_spikes'])
            analysis_data['sparsity'].extend(game_analysis['sparsity'])
            analysis_data['decision_times'].extend(game_analysis['decision_times'])
            analysis_data['ac_counts'].extend(game_analysis['ac_counts'])
            analysis_data['mac_counts'].extend(game_analysis['mac_counts'])
            analysis_data['internal_mac_counts'].extend(game_analysis['internal_mac_counts'])

    # Print game results
    print(f"\n{agent_name} vs. Random Agent - Final Results (Agent as X)")
    print("="*50)
    print(f"Wins ({agent_name}): {results['agent']}, Losses: {results['random']}, Draws: {results['draw']}")
    print(f"Win rate: {results['agent'] / num_games:.2%}, "
          f"Loss rate: {results['random'] / num_games:.2%}, "
          f"Draw rate: {results['draw'] / num_games:.2%}")

    # Print analysis results
    if isinstance(agent, DSQN) and analysis_data['total_spikes']:
        print(f"\nDSQN (TTFS Encoding) Analysis Results")
        print("="*50)
        print(f"Average Total Spikes per Decision: {float(np.mean(analysis_data['total_spikes'])):.2f} "
              f"(Std: {float(np.std(analysis_data['total_spikes'])):.2f})")
        print(f"Average Sparsity per Decision: {float(np.mean(analysis_data['sparsity'])):.2%} "
              f"(Std: {float(np.std(analysis_data['sparsity'])):.2%})")
        print(f"Average ACs per Decision (spike-triggered additions): {float(np.mean(analysis_data['ac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['ac_counts'])):.2f})")
        print(f"Average Internal State Update MACs per Decision: {float(np.mean(analysis_data['internal_mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['internal_mac_counts'])):.2f})")
        decision_time_counts = np.bincount(analysis_data['decision_times'], minlength=agent.simulation_time + 1)[1:]
        print(f"Decision Stabilization Times (over all decisions):")
        for t in range(agent.simulation_time):
            print(f"  Time Step {t+1}: {decision_time_counts[t]} decisions "
                  f"({decision_time_counts[t] / len(analysis_data['decision_times']):.2%})")
        # Compute energy ratio and savings
        total_spikes_avg = float(np.mean(analysis_data['total_spikes']))
        total_neurons = 256  # 128 + 128 (hidden layers)
        total_possible_spikes = total_neurons * agent.simulation_time  # 256 * 5
        f_r = total_spikes_avg / total_possible_spikes
        avg_ac = float(np.mean(analysis_data['ac_counts']))
        avg_internal_mac = float(np.mean(analysis_data['internal_mac_counts']))
        ann_energy = 18688 * 31  # ANN MACs * E_MAC (E_AC = 1)
        snn_energy = avg_ac + avg_internal_mac * 31  # E_SNN = AC * E_AC + Internal_MAC * E_MAC
        energy_ratio_detailed = snn_energy / ann_energy
        energy_savings_detailed = (1 - energy_ratio_detailed) * 100
        energy_ratio_simplified = agent.simulation_time * f_r * (1 / 31)
        energy_savings_simplified = (1 - energy_ratio_simplified) * 100
        print(f"Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): {energy_ratio_simplified:.4f}")
        print(f"Simplified Energy Savings: {energy_savings_simplified:.1f}%")
        print(f"Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): {energy_ratio_detailed:.4f}")
        print(f"Detailed Energy Savings: {energy_savings_detailed:.1f}%")
    elif isinstance(agent, DQN) and analysis_data['mac_counts']:
        print(f"\nDQN Analysis Results")
        print("="*50)
        print(f"Average MACs per Decision: {float(np.mean(analysis_data['mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['mac_counts'])):.2f})")

if __name__ == "__main__":
    # Set random seeds for reproducibility
    random.seed(82)
    np.random.seed(82)
    torch.manual_seed(82)

    # DSQN configuration (same as TTFS training code)
    dsnn_config = {
        'architecture': [9, 128, 128, 9],  # Input: 9 neurons (one per board position)
        'seed': 82,
        'alpha': 0.9,
        'beta': 0.85,
        'weight_scale': 0.15,
        'batch_size': 32,
        'threshold': 0.1,
        'simulation_time': 5,
        'learning_rate': 0.0001,
        'reset_potential': 0.0
    }

    # Initialize DQN and DSQN
    dqn_agent = DQN()
    dsqn_agent = DSQN(dsnn_config=dsnn_config)

    print("\nTesting DQN vs. Random Agent (Agent as X)")
    test_agent_vs_random(dqn_agent, "DQN", num_games=100, visualize_all=False)
    
    print("\nTesting DSQN (TTFS Encoding) vs. Random Agent (Agent as X)")
    test_agent_vs_random(dsqn_agent, "DSQN", num_games=100, visualize_all=False)



Testing DQN vs. Random Agent (Agent as X)
DQN model loaded from ../saved_models/ttt_dqn_first_final.pth

DQN vs. Random Agent - Final Results (Agent as X)
Wins (DQN): 97, Losses: 0, Draws: 3
Win rate: 97.00%, Loss rate: 0.00%, Draw rate: 3.00%

DQN Analysis Results
Average MACs per Decision: 18688.00 (Std: 0.00)

Testing DSQN (TTFS Encoding) vs. Random Agent (Agent as X)
DSQN model loaded from ../saved_models/ttfs_ttt.pth

DSQN vs. Random Agent - Final Results (Agent as X)
Wins (DSQN): 99, Losses: 0, Draws: 1
Win rate: 99.00%, Loss rate: 0.00%, Draw rate: 1.00%

DSQN (TTFS Encoding) Analysis Results
Average Total Spikes per Decision: 146.10 (Std: 86.23)
Average Sparsity per Decision: 88.59% (Std: 6.74%)
Average ACs per Decision (spike-triggered additions): 8768.60 (Std: 4148.13)
Average Internal State Update MACs per Decision: 2942.20 (Std: 172.45)
Decision Stabilization Times (over all decisions):
  Time Step 1: 38 decisions (12.14%)
  Time Step 2: 137 decisions (43.77%)
  Time Step 